# Why use GLIDE ? Evaluate any risk, of any AI system

> **From noisy LLM judges to statistically rigorous quality metrics — in minutes.**

**GLIDE** — the *Generative Label Inference & Debiasing Engine* — combines a small set of expensive true evaluation labels with a large pool of cheap proxy evaluation labels to produce a statistically valid, bias-corrected quality metric. This is necessary because proxy evaluation is generally carried out using LLM judges who are known to disagree between one another. This guide walks through a realistic hallucination-detection scenario end-to-end.

---

**What you will learn:**

- Why LLM-as-Judge metrics are systematically biased
- How to use GLIDE to produce a bias-corrected metric
- How to test that your metric fits your expectations

## The problem: your LLM judge disagrees with your users

Let's assume you run a customer-facing agentic assistant handling thousands of conversations per day.

### The signals

- **Every fifth user** reports incorrect or fabricated information (unacceptable for the management).
- You deploy an LLM judge to measure the hallucination rate. The latter tells you the hallucination rate is **6%**. 

The users and the LLM judge **disagree**. You decide to dig deeper.

### The manual investigation

You budget for **100 manual annotations** — expensive but accurate ground truth.
Annotators find that **~12% of conversations contain a blatant hallucination**.

That is **double what LLM reports**. The judge is *systematically optimistic*.

LLM is closer to the annotators so you decide to continue with it, but it appears to be biased.

### The challenge

You now have:
- **1,000 LLM judgements** — cheap and fast, but biased
- **100 human annotations** — accurate, but covering only 10% of your data

**GLIDE combines both** to produce a reliable, unbiased estimate of the true hallucination rate across all 1,000 conversations.

![Methodology](https://raw.githubusercontent.com/EmertonData/glide/refs/heads/main/docs/assets/schema-PPI.png)
*A large number of interactions with an AI system are collected. A large subset is labeled by an LLM Judge only and a small subset is labeled by both the LLM judge and human annotators*


In [ ]:
%matplotlib inline

import json

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from glide.core.simulated import generate_dataset_binary
from glide.estimators.ppi import PPIMeanEstimator

# ── Colour palette ──────────────────────────────────────────
C_JUDGE = '#E74C3C'   # LLM judge  — red-orange
C_HUMAN = '#2E86AB'   # Human-only — blue
C_GLIDE = '#27AE60'   # GLIDE      — green
C_TRUTH = '#2C3E50'   # True value — dark slate

# ── Global plot style ───────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor'  : '#FAFAFA',
    'axes.grid'       : True,
    'grid.color'      : '#E5E5E5',
    'grid.linewidth'  : 0.8,
    'axes.titlesize'  : 14,
    'axes.titlepad'   : 14,
    'axes.labelsize'  : 12,
    'xtick.labelsize' : 11,
    'ytick.labelsize' : 11,
})

## Simulating 1,000 conversations with a biased judge

`generate_dataset_binary` produces a synthetic dataset that mirrors the scenario above.

Each **record** represents one conversation:

| Field | Present for | Meaning |
|-------|-------------|---------|
| `y_true` | Labeled records only | `1` = hallucination confirmed by human annotator |
| `y_proxy` | All records | `1` = hallucination flagged by the LLM judge |

Parameters match the scenario from section 1:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `n` | 100 | Manually annotated conversations |
| `N` | 900 | LLM-judged only (no human label) |
| `true_mean` | 0.10 | True hallucination rate: **10%** |
| `proxy_mean` | 0.05 | LLM judge rate: **5%** (underestimates) |
| `correlation` | 0.9 | Pearson correlation between Judge and truth |

In [ ]:
dataset = generate_dataset_binary(
    n=100,            
    N=900,            
    true_mean=0.10,   
    proxy_mean=0.05,  
    correlation=0.65, 
    random_seed=4242,
)

labeled   = [r for r in dataset if 'y_true'  in r]
unlabeled = [r for r in dataset if 'y_true' not in r]


In [ ]:
print(f'Total conversations  : {len(dataset):,}')
print(f'  Manually annotated : {len(labeled):,}')
print(f'  LLM-judged only    : {len(unlabeled):,}')
print()
print('Sample records')
print(f'  labeled   -> {labeled[0]}')
print(f'  unlabeled -> {unlabeled[0]}')

## Two naive strategies both fail

Two obvious approaches to estimating the true hallucination rate each have a fatal flaw:

**Option A — Trust the judge on all 1,000 conversations.**  
Precise (large sample), but the judge's systematic bias makes the estimate wrong.

**Option B — Trust only the 100 human annotations.**  
Unbiased, but the 95% confidence interval (CI) is very wide — 100 samples is not enough for a precise estimate.

GLIDE, introduced in the next section, fixes both problems simultaneously.

In [ ]:
z95 = norm.ppf(0.975)

# Option A: LLM judge — average proxy labels over all 1,000 conversations
proxy_all  = [r['y_proxy'] for r in dataset]
judge_mean = np.mean(proxy_all)
judge_se   = np.std(proxy_all, ddof=1) / np.sqrt(len(proxy_all))

# Option B: human labels only — average over 100 annotated conversations
true_vals  = [r['y_true'] for r in labeled]
human_mean = np.mean(true_vals)
human_se   = np.std(true_vals, ddof=1) / np.sqrt(len(true_vals))


In [ ]:
sep = '-' * 62
print(sep)
print(f'{"Method":<34} {"Estimate":>8}   {"95% CI":>16}')
print(sep)
print(
    f'{"LLM Judge only  (n=1,000)":<34} {judge_mean:>7.1%}   '
    f'[{judge_mean - z95*judge_se:.1%}, {judge_mean + z95*judge_se:.1%}]'
)
print(
    f'{"Human labels only (n=100)":<34} {human_mean:>7.1%}   '
    f'[{human_mean - z95*human_se:.1%}, {human_mean + z95*human_se:.1%}]'
)
print(sep)
print(f'{"True rate  (simulation)":<34} {"10.0%":>8}')

## The root cause: the LLM judge is systematically biased

The gap is clear: compared to human annotators, the judge consistently under-reports hallucinations on average.

GLIDE's **PPI rectifier** measures this systematic error on the labeled subset, then applies the correction to the remaining 900 conversations:

$$\hat{\mu}_{\text{PPI}} = \underbrace{\bar{y}_{\text{proxy}}^{\,(N)}}_\text{LLM-as-judge} + \underbrace{\left(\bar{y}_{\text{true}} - \bar{y}_{\text{proxy}}^{\,(n)}\right)}_\text{rectifier}$$

where : 

$\bar{y}_{\text{proxy}}^{\,(N)} = $ proxy mean computed on $N$ unlabeled samples

$\bar{y}_{\text{proxy}}^{\,(n)} = $ proxy mean computed on $n$ labeled samples

$\bar{y}_{\text{true}} = $ true mean estimate computed on $n$ labeled samples

The rectifier ($\bar{y}_{\text{true}} - \bar{y}_{\text{proxy}}^{\,(n)}$) equals zero when the judge is perfectly calibrated and grows large when the judge is biased — exactly the situation here.

In [ ]:
proxy_on_labeled = [r['y_proxy'] for r in labeled]
true_on_labeled  = [r['y_true']  for r in labeled]

p_mean = np.mean(proxy_on_labeled)
t_mean = np.mean(true_on_labeled)
bias   = p_mean - t_mean

fig, ax = plt.subplots(figsize=(7, 4.5))

x_pos      = np.array([0, 1])
bar_vals   = [p_mean, t_mean]
bar_colors = [C_JUDGE, C_HUMAN]
bar_labels = ['LLM Judge\n on annotated subset', 
              'Human Annotation\n(ground truth)']

ax.bar(x_pos, bar_vals, color=bar_colors, width=0.45,
       edgecolor='white', linewidth=2, zorder=3)

for xi, val, c in zip(x_pos, bar_vals, bar_colors):
    ax.text(xi, val + 0.005, f'{val:.1%}',
            ha='center', fontsize=14, fontweight='bold', color=c)

ax.annotate('', xy=(1, t_mean + 0.010), xytext=(0, p_mean + 0.010),
            arrowprops=dict(arrowstyle='<->', color='#666666', lw=2.5))
ax.text(0.5, max(bar_vals) + 0.033, f'Bias = {bias:+.1%}',
        ha='center', fontsize=12, color='#555555', fontstyle='italic',
        bbox=dict(boxstyle='round,pad=0.35', 
                  facecolor='#FFFDE7', edgecolor='#CCCCCC'))

ax.axhline(0.10, color=C_TRUTH, linestyle='--', 
           linewidth=1.8, zorder=2, label='True rate: 10%')

ax.set_xticks(x_pos)
ax.set_xticklabels(bar_labels, fontsize=11)
ax.set_ylabel('Hallucination Rate', fontsize=12)
ax.set_title('LLM Judge Systematically Underestimates Hallucinations',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 0.26)
ax.set_xlim(-0.5, 1.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

## GLIDE corrects the bias using Prediction-Powered Inference

`PPIMeanEstimator` implements Prediction-Powered Inference (PPI)
([Angelopoulos et al., *Science* 2023](https://www.science.org/doi/10.1126/science.adi6000)).

PPI produces a **bias-corrected point estimate** with a **tighter confidence interval (CI)**
than either data source alone, by combining:

- **n = 100** labeled records (with human annotations — expensive ground truth — and LLM judge labels)
- **N = 900** unlabeled records (LLM judge labels — cheap proxies)

The standard error (SE) of the PPI estimate decomposes into two terms:

$$\text{SE}_{\text{PPI}} = \underbrace{\sqrt{\frac{\text{Var}(y_{\text{true}} - y_{\text{proxy}})}{n}}}_{\text{rectifier noise}} + \underbrace{\sqrt{\frac{\text{Var}(y_{\text{proxy}}^{\,N})}{N}}}_{\text{proxy noise}}$$

**When judge and human labels are correlated**, the difference $y_{\text{true}} - y_{\text{proxy}}$ has low variance — the rectifier adds little noise. GLIDE achieves the **precision of 1,000 labels** while **paying for only 100**.

In [ ]:
estimator = PPIMeanEstimator()

result = estimator.estimate(
    dataset,
    y_true_field='y_true',
    y_proxy_field='y_proxy',
    metric_name='Hallucination Rate',
    confidence_level=0.95,
)

print(result.summary())

### Exporting the Result as JSON

The `InferenceResult` object serializes directly to a plain dictionary, ready for logging, dashboards, or downstream pipelines.

In [ ]:
result_dict = {
    'metric_name'         : result.metric_name,
    'estimator'           : result.estimator_name,
    'n_labeled'           : result.n_true,
    'n_total'             : result.n_proxy,
    'point_estimate'      : round(result.result.mean, 4),
    'confidence_interval' : {
        'lower'            : round(result.result.lower_bound, 4),
        'upper'            : round(result.result.upper_bound, 4),
        'confidence_level' : result.result.confidence_level,
    },
    'standard_error'      : round(result.result.std, 4),
}

print(json.dumps(result_dict, indent=2))

## GLIDE Delivers an Unbiased Estimate at Low Cost

The plot below compares **point estimates** and **95% confidence intervals** for all three methods against the true hallucination rate (dashed line):

- **LLM judge**: very narrow CI, but **wrong** — points at 5%, not 10%.
- **Human-only**: unbiased, but the CI is very wide — 100 samples is not enough for precision.
- **GLIDE**: unbiased *and* narrow — the accuracy of human labels combined with the precision of 1,000 proxy judgements.

In [ ]:
TRUE_RATE = 0.10

estimates = [
    (
        'LLM Judge\n(n=1,000  |  raw proxy)',
        judge_mean,
        judge_mean - z95 * judge_se,
        judge_mean + z95 * judge_se,
        C_JUDGE,
    ),
    (
        'Human Annotation\n(n=100  |  small sample)',
        human_mean,
        human_mean - z95 * human_se,
        human_mean + z95 * human_se,
        C_HUMAN,
    ),
    (
        'GLIDE — PPI\n(n=100  +  N=900)',
        result.result.mean,
        result.result.lower_bound,
        result.result.upper_bound,
        C_GLIDE,
    ),
]

fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = [2, 1, 0]

for y, (label, mean, lo, hi, color) in zip(y_pos, estimates):
    # CI line
    ax.plot([lo, hi], [y, y], color=color, linewidth=4,
            solid_capstyle='round', zorder=3)
    # Cap marks
    for xc in [lo, hi]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2],
                color=color, linewidth=2.5, zorder=3)
    # Point estimate
    ax.scatter(mean, y, s=200, color=color, zorder=5,
               edgecolors='white', linewidths=2.5)
    # Value label above
    ax.text(mean, y + 0.34, f'{mean:.1%}',
            ha='center', va='bottom', fontsize=12,
            color=color, fontweight='bold')
    # CI bounds below
    ax.text(mean, y - 0.34, f'[{lo:.1%},  {hi:.1%}]',
            ha='center', va='top', fontsize=11, color='#888888')

# True value
ax.axvline(TRUE_RATE, color=C_TRUTH, 
           linestyle='--', linewidth=2.5, zorder=4)
ax.text(TRUE_RATE + 0.004, 2.72, 'True rate  10%',
        color=C_TRUTH, fontsize=10.5, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in estimates], fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_xlabel('Estimated Hallucination Rate', fontsize=12)
ax.set_title('GLIDE Delivers an Unbiased, Precise Estimate',
             fontsize=14, fontweight='bold')
ax.set_xlim(-0.01, 0.26)
ax.set_ylim(-0.8, 3.2)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Testing whether the hallucination rate is within acceptable limits

GLIDE's estimate enables a formal hypothesis test: is the true hallucination rate significantly higher than the 5% LLM reported?

$$H_0 : \mu = 5\% \qquad H_1 : \mu > 5\%$$

Without GLIDE, this test is impossible — relying on the judge alone, the point estimate *is* 5%, so the null hypothesis trivially holds. GLIDE provides the statistical power to detect the discrepancy.

In [ ]:
z_stat, p_value, _ = result.result.test_null_hypothesis(
    h0_value=0.05,       # LLM judge's claimed rate
    alternative='larger' # H1: true rate > 5%
)

sep = '-' * 48
print('Hypothesis test')
print(sep)
print('H0 : hallucination rate = 5%   (LLM says so)')
print('H1 : hallucination rate > 5%   (users complain!)')
print()
print(f'z-statistic : {z_stat:.2f}')
print(f'p-value     : {p_value:.4f}')
print()
if p_value < 0.05:
    print('Decision  : Reject H0 at the 5% level.')
    print('=> The true hallucination rate is significantly above 5%.')
else:
    print('Decision  : Cannot reject H0 at the 5% level.')

## Summary: GLIDE combines accuracy and precision

| | LLM Judge | Human-only | **GLIDE** |
|-|-----------|-----------|-----------|
| Sample size | 1,000 | 100 | 1,000 |
| Unbiased estimate | ❌ | ✅ | ✅ |
| Narrow confidence interval | 🟠 *(misleading)* | ❌ | ✅ |
| Annotation cost | Low | **High** | Small |

**Key takeaways:**

1. **LLM judges are biased.** A narrow CI around the wrong value is worse than useless — it gives false confidence.

2. **100 human annotations is all you need.** The rectifier uses information from 900 cheap proxy labels to shrink the confidence interval by ~50% compared to human-only estimation.

3. **PPI efficiency relies on sample size and LLM-judge quality.** In this run, the 100 annotated conversations happened to show a 12–13% hallucination rate. GLIDE's estimate is better than the judge alone. The null hypothesis (μ = 5%) cannot be rejected — more human annotations or a better LLM-judge would help narrow the CI further.

---

*Want to go further? Knowing precisely what you are measuring — and with what uncertainty — is the essential first step into LLM alignment, guardrailing, and agentic workflows with self-evaluation loops. These next steps will soon be included in GLIDE.* 